# Training LDA model

In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [41]:
df = pd.read_csv("data/merged_data.csv", index_col="date", parse_dates=True)
df.sort_index(inplace=True)

We define our features (X) and target (y)

Our features (X) is comprised of our cleaned text data (X1) and our financial data (X2).

In [42]:
X = df[["summary", "pct_change", "volume"]]
y = df["target"]

Now we split our data for training and testing

DO NOT SHUFFLE because we don't want to mix time series data

In [43]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [44]:
print(f"Training Range: {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing Range:  {X_test.index.min()} to {X_test.index.max()}")

Training Range: 2017-12-19 00:00:00 to 2020-01-10 00:00:00
Testing Range:  2020-01-13 00:00:00 to 2020-07-17 00:00:00


Now we have two different types of data that we need to handle differently

1. Text data - we will use TF-IDF vectorization to convert text into numerical format
2. Numerical data - we will use StandardScaler or MinMaxScaler to normalize the data

The reason we need two different scalers is because Naive Bayes won't work with negative values

In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import TruncatedSVD

In [46]:
text_features = "summary"
numerical_features = ["pct_change", "volume"]

Make our preprocessor for the vectorization and normalization steps

In [47]:
preprocessor_lda = ColumnTransformer(
    transformers=[
        # Reduce text to 50 components using SVD (PCA for text)
        (
            "text",
            Pipeline(
                [
                    ("tfidf", TfidfVectorizer(max_features=5000)),
                    ("svd", TruncatedSVD(n_components=50)),
                ]
            ),
            text_features,
        ),
        ("num", StandardScaler(), numerical_features),
    ]
)

Setup our pipeline with the SGDClassifier model

In [48]:
pipeline_lda = Pipeline(
    [("preprocessor", preprocessor_lda), ("classifier", LinearDiscriminantAnalysis())]
)

And fit the model

In [49]:
pipeline_lda.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


That trained way too fast... we probably aren't using enough data.

Let's see how it performs

In [50]:
y_pred = pipeline_lda.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.6384615384615384
              precision    recall  f1-score   support

         0.0       0.62      0.50      0.55        58
         1.0       0.65      0.75      0.70        72

    accuracy                           0.64       130
   macro avg       0.63      0.62      0.62       130
weighted avg       0.64      0.64      0.63       130



In [51]:
%%capture
%pip install joblib

In [52]:
# exporting the baseline model to compare after we tune it
import joblib

joblib.dump(pipeline_lda, "models/lda_baseline.pkl")

['models/lda_baseline.pkl']